# RelaLeap Colab Phase 0 Comparison

This notebook is a thin Colab wrapper around the GitHub repo. The repo, configs, baseline, and run artifacts are the source of truth.

Before running, choose `Runtime -> Change runtime type -> GPU` if a GPU run is desired.

In [ ]:
!nvidia-smi || true
import torch, platform
print('platform:', platform.platform())
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
repo = Path('/content/relaleap')
if repo.exists():
    %cd /content/relaleap
    !git pull --ff-only
else:
    %cd /content
    !git clone https://github.com/bgoertzel-sing/relaleap.git
    %cd /content/relaleap

In [ ]:
!python -m pip install -q -e '.[notebook]'
!python -m relaleap.experiments.compare --out results/comparisons/colab_phase0 --baseline-reference baselines/phase0_char_smoke_comparison.json
!python -m relaleap.experiments.run --config configs/char_smoke_pinned_hep.yaml --out results/runs/colab_char_smoke_pinned_hep
!python -m relaleap.experiments.compare --config configs/char_smoke_hep_support_stress.yaml --config configs/char_smoke_pinned_hep_support_stress.yaml --out results/comparisons/colab_support_stress_pinned_vs_repicked
!python -m relaleap.experiments.check_artifacts --comparison-dir results/comparisons/colab_support_stress_pinned_vs_repicked --out results/comparisons/colab_support_stress_pinned_vs_repicked/artifact_check.json
!python -m relaleap.experiments.compare --config configs/char_smoke_hep_support_stress.yaml --config configs/char_smoke_hep_support_stress_clipped.yaml --out results/comparisons/colab_support_stress_clipped_hep
!python -m relaleap.experiments.check_artifacts --comparison-dir results/comparisons/colab_support_stress_clipped_hep --out results/comparisons/colab_support_stress_clipped_hep/artifact_check.json

In [ ]:
import base64
import io
import json
import textwrap
import zipfile
from pathlib import Path
summary_path = Path('results/comparisons/colab_phase0/summary.json')
baseline_comparison_path = Path('results/comparisons/colab_phase0/baseline_comparison.json')
pinned_summary_path = Path('results/runs/colab_char_smoke_pinned_hep/summary.json')
support_stress_summary_path = Path('results/comparisons/colab_support_stress_pinned_vs_repicked/summary.json')
support_stress_check_path = Path('results/comparisons/colab_support_stress_pinned_vs_repicked/artifact_check.json')
clipped_summary_path = Path('results/comparisons/colab_support_stress_clipped_hep/summary.json')
clipped_check_path = Path('results/comparisons/colab_support_stress_clipped_hep/artifact_check.json')
print(summary_path.read_text())
print(baseline_comparison_path.read_text())
print(pinned_summary_path.read_text())
print(support_stress_summary_path.read_text())
print(support_stress_check_path.read_text())
print(clipped_summary_path.read_text())
print(clipped_check_path.read_text())
summary = json.loads(summary_path.read_text())
baseline_comparison = json.loads(baseline_comparison_path.read_text())
pinned_summary = json.loads(pinned_summary_path.read_text())
support_stress_summary = json.loads(support_stress_summary_path.read_text())
support_stress_check = json.loads(support_stress_check_path.read_text())
clipped_summary = json.loads(clipped_summary_path.read_text())
clipped_check = json.loads(clipped_check_path.read_text())
assert summary['status'] == 'ok'
assert summary['verdict']['status'] == 'pass'
assert baseline_comparison['status'] == 'pass'
assert pinned_summary['status'] == 'ok'
assert pinned_summary['phase0']['pinned_support'] is True
assert pinned_summary['phase0']['invariants']['hep_alpha_0_equivalence'] is True
assert support_stress_summary['status'] == 'ok'
assert support_stress_summary['verdict']['status'] == 'pass'
assert support_stress_check['status'] == 'pass'
support_stress_runs = {run['experiment_id']: run for run in support_stress_summary['runs']}
assert support_stress_runs['char_smoke_hep_support_stress']['support_stress'] is True
assert support_stress_runs['char_smoke_pinned_hep_support_stress']['pinned_support'] is True
assert support_stress_runs['char_smoke_pinned_hep_support_stress']['support_stress'] is True
assert clipped_summary['status'] == 'ok'
assert clipped_summary['verdict']['status'] == 'pass'
assert clipped_check['status'] == 'pass'
clipped_runs = {run['experiment_id']: run for run in clipped_summary['runs']}
assert clipped_runs['char_smoke_hep_support_stress_clipped']['hep_update_clip_norm'] == 0.01
assert clipped_runs['char_smoke_hep_support_stress_clipped']['support_stress'] is True
accepted = summary['verdict']['hep_alpha_acceptance']['accepted_alpha']
print('Accepted HEP alpha:', accepted)
print('Pinned HEP status:', pinned_summary['status'])
print('Support-stress comparison status:', support_stress_summary['status'])
print('Support-stress artifact check:', support_stress_check['status'])
print('Clipped support-stress comparison status:', clipped_summary['status'])
print('Clipped support-stress artifact check:', clipped_check['status'])
artifact_roots = [
    Path('results/comparisons/colab_phase0'),
    Path('results/runs/colab_char_smoke_pinned_hep'),
    Path('results/comparisons/colab_support_stress_pinned_vs_repicked'),
    Path('results/comparisons/colab_support_stress_clipped_hep'),
]
bundle_buffer = io.BytesIO()
with zipfile.ZipFile(bundle_buffer, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact_root in artifact_roots:
        for path in sorted(artifact_root.rglob('*')):
            if path.is_file():
                archive.write(path, path.as_posix())
bundle = base64.b64encode(bundle_buffer.getvalue()).decode('ascii')
print('RELALEAP_ARTIFACT_BUNDLE_ZIP_BASE64_BEGIN')
print('\n'.join(textwrap.wrap(bundle, width=120)))
print('RELALEAP_ARTIFACT_BUNDLE_ZIP_BASE64_END')
print('RelaLeap Colab Phase 0 comparison completed.')